# Code for getting forced liquidation order data for tickers given

In [5]:
import asyncio
import json
import os
from websockets import connect
import pandas as pd

# ...existing code...
def run_liquidation_logger(tickers, filename="binance_liq1.csv"):
    """
    Start a websocket that logs liquidation data for a list of symbols.

    tickers  : list of base symbols like ["BTC", "ETH", "SOL"]
    filename : csv file to append to
    """

    # Build combined-stream URI  e.g.
    # wss://fstream.binance.com/stream?streams=btcusdt@forceOrder/ethusdt@forceOrder
    streams = "/".join(f"{ticker.lower()}usdt@forceOrder" for ticker in tickers)
    # print(streams) # debug
    uri = f"wss://fstream.binance.com/stream?streams={streams}"

    # Create file & header if it doesn't exist
    if not os.path.isfile(filename):
        with open(filename, "w") as f:
            header = [
                "symbol","side","order_type","time_in_force",
                "original_quantity","price","average_price",
                "order_status","order_last_filled_quantity",
                "order_filled_accumalated_quantity","order_trade_time"
            ]
            f.write(",".join(header) + "\n")

    async def _collect():
        while True:
            try:
                # Use context manager and iterate messages
                async with connect(uri) as websocket:
                    async for raw in websocket:
                        data = json.loads(raw)
                        # each message contains {"stream": "...", "data": {...}}
                        msg = data["data"]["o"]   # order object
                        line = ",".join(str(v) for v in msg.values())
                        with open(filename, "a") as f:
                            f.write(line + "\n")
            except Exception as e:
                print("Connection error:", e)
                # short delay before reconnecting
                await asyncio.sleep(1)

    loop = asyncio.get_event_loop()
    if loop.is_running():
        # schedule background task in notebook's event loop
        asyncio.create_task(_collect())
    else:
        # normal script usage
        loop.run_until_complete(_collect())
# ...existing code...
univ = ['BTC','ETH','SOL','XRP','LTC','ADA','DOGE','DOT','MATIC','AVAX']
run_liquidation_logger(univ)  # starts background task in Jupyter
# ...existing code...

In [6]:
data = pd.read_csv("binance_liq1.csv")
data

,symbol,side,order_type,time_in_force,original_quantity,price,average_price,order_status,order_last_filled_quantity,order_filled_accumalated_quantity,order_trade_time
0,XRPUSDT,SELL,LIMIT,IOC,692.500,2.96220,2.97700,FILLED,692.500,692.500,1759810405263
1,DOGEUSDT,SELL,LIMIT,IOC,3077.000,0.26314,0.26481,FILLED,3077.000,3077.000,1759810426064
2,ETHUSDT,BUY,LIMIT,IOC,0.024,4697.38000,4678.77000,FILLED,0.024,0.024,1759810429956
3,DOGEUSDT,SELL,LIMIT,IOC,4652.000,0.26307,0.26480,FILLED,2395.000,4652.000,1759810431438
4,BTCUSDT,SELL,LIMIT,IOC,0.002,123617.40000,124097.20000,FILLED,0.002,0.002,1759810432254
5,SOLUSDT,SELL,LIMIT,IOC,0.540,231.84000,233.00000,FILLED,0.540,0.540,1759810432796
6,DOTUSDT,SELL,LIMIT,IOC,54.000,4.30700,4.33400,FILLED,6.800,54.000,1759810436688
7,BTCUSDT,SELL,LIMIT,IOC,0.004,123600.80000,124097.20000,FILLED,0.002,0.004,1759810438152
8,SOLUSDT,SELL,LIMIT,IOC,1.330,231.83000,232.95000,FILLED,1.330,1.330,1759810438152
9,LTCUSDT,SELL,LIMIT,IOC,0.246,117.32000,117.88000,FILLED,0.139,0.246,1759810438709


In [7]:
data["order time"] = pd.to_datetime(data["order_trade_time"], unit='ms', utc=True).dt.tz_convert('US/Pacific')
data.drop(columns=["order_type"], inplace=True)
data.drop(columns=["original_quantity"], inplace=True)
data.drop(columns=["price"], inplace=True)
data.drop(columns=["order_status"], inplace=True)
data.drop(columns=["order_last_filled_quantity"], inplace=True)
data['notional'] = data['average_price'].astype(float) * data['order_filled_accumalated_quantity'].astype(float)
data

,symbol,side,time_in_force,average_price,order_filled_accumalated_quantity,order_trade_time,order time,notional
0,XRPUSDT,SELL,IOC,2.97700,692.500,1759810405263,2025-10-06 21:13:25.263000-07:00,2061.57250
1,DOGEUSDT,SELL,IOC,0.26481,3077.000,1759810426064,2025-10-06 21:13:46.064000-07:00,814.82037
2,ETHUSDT,BUY,IOC,4678.77000,0.024,1759810429956,2025-10-06 21:13:49.956000-07:00,112.29048
3,DOGEUSDT,SELL,IOC,0.26480,4652.000,1759810431438,2025-10-06 21:13:51.438000-07:00,1231.84960
4,BTCUSDT,SELL,IOC,124097.20000,0.002,1759810432254,2025-10-06 21:13:52.254000-07:00,248.19440
5,SOLUSDT,SELL,IOC,233.00000,0.540,1759810432796,2025-10-06 21:13:52.796000-07:00,125.82000
6,DOTUSDT,SELL,IOC,4.33400,54.000,1759810436688,2025-10-06 21:13:56.688000-07:00,234.03600
7,BTCUSDT,SELL,IOC,124097.20000,0.004,1759810438152,2025-10-06 21:13:58.152000-07:00,496.38880
8,SOLUSDT,SELL,IOC,232.95000,1.330,1759810438152,2025-10-06 21:13:58.152000-07:00,309.82350
9,LTCUSDT,SELL,IOC,117.88000,0.246,1759810438709,2025-10-06 21:13:58.709000-07:00,28.99848


In [8]:
data.drop(columns=["order_trade_time"], inplace=True)

In [10]:
import yfinance as yf

px = yf.download(univ, start="2025-10-6", interval="1m", auto_adjust=False)['Adj Close']  # for n  ow the time frame is last sunday 
ret = px / px.shift() - 1
ret

[**********************70%*********              ]  7 of 10 completedHTTP Error 404: 
HTTP Error 404: 
[*********************100%***********************]  10 of 10 completed

6 Failed downloads:
['ADA', 'DOT', 'XRP']: YFPricesMissingError('possibly delisted; no price data found  (1m 2025-10-6 -> 2025-10-07 00:18:23-04:00)')
['MATIC', 'AVAX', 'DOGE']: YFTzMissingError('possibly delisted; no timezone found')


Ticker,ADA,AVAX,BTC,DOGE,DOT,ETH,LTC,MATIC,SOL,XRP
2025-10-06 13:30:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-10-06 13:31:00+00:00,NaN,NaN,-0.000181,NaN,NaN,-0.000924,0.002084,NaN,0.000000,NaN
2025-10-06 13:32:00+00:00,NaN,NaN,-0.000724,NaN,NaN,-0.000300,NaN,NaN,0.000000,NaN
2025-10-06 13:33:00+00:00,NaN,NaN,0.000724,NaN,NaN,-0.000969,NaN,NaN,NaN,NaN
2025-10-06 13:34:00+00:00,NaN,NaN,-0.000181,NaN,NaN,0.001734,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2025-10-06 19:55:00+00:00,NaN,NaN,0.000000,NaN,NaN,0.001580,-0.001421,NaN,0.000000,NaN
2025-10-06 19:56:00+00:00,NaN,NaN,0.000180,NaN,NaN,0.000563,-0.000711,NaN,0.000000,NaN
2025-10-06 19:57:00+00:00,NaN,NaN,0.000180,NaN,NaN,0.000450,-0.000427,NaN,0.000000,NaN
2025-10-06 19:58:00+00:00,NaN,NaN,0.000450,NaN,NaN,0.002813,0.000000,NaN,0.000000,NaN


In [ ]:
# Action Item: Need to organize data in format of rets and the singular data we are looking at is liquidational notional values, the rows are time periods and the columns are the different tickers

### What is k?

In [ ]:
# Action Item: scans for time period with liquidational notionals spikes for each ticker by first diving into k